In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from xgboost import XGBClassifier

In [2]:
# Carregamento dos dados

df = pd.read_csv(
    r'c:\Users\mathe\Downloads\banco_ia.csv',
    dtype=str,
    low_memory=False
)

df = df.drop(columns=[
    'ctrc_nota_fiscal', 'ctrc', 'nro_nota_fiscal',
    'data_emissao', 'cod_ocor'
], errors='ignore')

In [3]:
# Pré-processamento: CEP e CNPJs

# CEP: extrai os 2 primeiros dígitos como proxy de região geográfica
# Ex: '36540-000' -> '36' (faixa de CEP da região)
if 'cep_entrega' in df.columns:
    df['cep_regiao'] = df['cep_entrega'].str[:2]
    df = df.drop(columns=['cep_entrega'])

# CNPJs: são identificadores, não magnitudes — dropar para evitar ruído
cnpj_cols = ['cnpj_destinatario', 'cnpj_remetente', 'cnpj_pagador']
df = df.drop(columns=[c for c in cnpj_cols if c in df.columns])

In [4]:
# Encoding

one_hot_columns = [
    'unidade_emissora', 'unidade_receptora', 'tipo_documento',
    'praca_expedidora', 'unidade_emissora.1', 'setor_coleta',
    'praca_destino', 'local_entrega', 'cidade_entrega',
    'uf_entrega', 'unidade_receptora.1', 'tipo_frete',
    'tipo_baixa', 'mercadoria', 'especie', 'modalidade',
    'segmento', 'cidade_origem_prestacao', 'rota_entrega',
    'endereco', 'bairro', 'placa_coleta', 'placa_entrega',
    'placa_cavalo', 'placa_carreta', 'ref_rota'
]

# Filtra apenas colunas existentes no df
one_hot_columns = [col for col in one_hot_columns if col in df.columns]

low_card_cols  = [col for col in one_hot_columns if df[col].nunique() <= 50]
high_card_cols = [col for col in one_hot_columns if df[col].nunique() > 50]

# Adiciona cep_regiao ao ordinal (baixa cardinalidade ~100 regiões)
if 'cep_regiao' in df.columns:
    high_card_cols.append('cep_regiao')

encoded_cols = set(low_card_cols + high_card_cols)

# Inspeciona colunas que irão para o remainder
remainder_cols = [col for col in df.columns if col not in encoded_cols]
print("Colunas no remainder (devem ser todas numéricas):", remainder_cols)

ct = ColumnTransformer(transformers=[
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False), low_card_cols),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), high_card_cols),
], remainder='passthrough')

df_transformed = ct.fit_transform(df)

# Nomes das colunas
onehot_names = ct.named_transformers_['onehot'].get_feature_names_out(low_card_cols)
passthrough_cols = [col for col in remainder_cols]  # ordem preservada pelo ColumnTransformer
all_column_names = list(onehot_names) + high_card_cols + passthrough_cols

df = pd.DataFrame(df_transformed, columns=all_column_names)

# Garante que tudo é numérico (converte strings residuais para NaN)
df = df.apply(pd.to_numeric, errors='coerce')

print(f"Shape após encoding: {df.shape}")
print(f"Colunas com dtype object restantes: {list(df.select_dtypes('object').columns)}")

Colunas no remainder (devem ser todas numéricas): ['peso_real_kg', 'cubagem_m3', 'quantidade_volumes', 'quantidade_pares', 'valor_liquidado', 'valor_mercadoria', 'valor_frete', 'peso_calculado_kg', 'frete_peso', 'frete_valor', 'historico_cod_ocor']
Shape após encoding: (617049, 140)
Colunas com dtype object restantes: []


In [5]:
# Target e Split

df = df.dropna(subset=['historico_cod_ocor'])

X = df.drop(columns=['historico_cod_ocor'])
y = df['historico_cod_ocor']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Distribuição do target:\n{y.value_counts()}")

X_train: (555344, 139) | X_test: (61705, 139)
Distribuição do target:
historico_cod_ocor
0     593045
6      15316
8       7103
9       1316
3        138
16       102
17        29
Name: count, dtype: int64


In [ ]:
# Random Forest

rf_params = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_features': ['sqrt', 'log2', None]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params,
    cv=3,
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)

rf_best = rf_grid.best_estimator_
print("Melhores params RF:", rf_grid.best_params_)

rf_preds = rf_best.predict(X_test)
print(f"RF — Accuracy:  {accuracy_score(y_test, rf_preds):.4f}")
print(f"RF — Precision: {precision_score(y_test, rf_preds, average='weighted', zero_division=0):.4f}")
print(f"RF — Recall:    {recall_score(y_test, rf_preds, average='weighted', zero_division=0):.4f}")

In [ ]:
# Extra Trees

et_params = {
    'criterion': ['gini', 'entropy'],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [False, True]
}

et_grid = GridSearchCV(
    ExtraTreesClassifier(random_state=5),
    param_grid=et_params,
    cv=3,
    n_jobs=-1,
    verbose=1
)

et_grid.fit(X_train, y_train)

et_best = et_grid.best_estimator_
print("Melhores params ET:", et_grid.best_params_)

et_preds = et_best.predict(X_test)
print(f"ET — Accuracy:  {accuracy_score(y_test, et_preds):.4f}")
print(f"ET — Precision: {precision_score(y_test, et_preds, average='weighted', zero_division=0):.4f}")
print(f"ET — Recall:    {recall_score(y_test, et_preds, average='weighted', zero_division=0):.4f}")

Fitting 3 folds for each of 432 candidates, totalling 1296 fits


In [ ]:
# XGBoost

xgb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 10],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.6, 0.8, 1],
    'colsample_bytree': [0.6, 0.8, 1],
    'gamma': [0, 1, 5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

xgb_grid = GridSearchCV(
    XGBClassifier(
        random_state=42,
        eval_metric='mlogloss'
    ),
    param_grid=xgb_params,
    cv=3,
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)

xgb_best = xgb_grid.best_estimator_
print("Melhores params XGB:", xgb_grid.best_params_)

xgb_preds = xgb_best.predict(X_test)
print(f"XGB — Accuracy:  {accuracy_score(y_test, xgb_preds):.4f}")
print(f"XGB — Precision: {precision_score(y_test, xgb_preds, average='weighted', zero_division=0):.4f}")
print(f"XGB — Recall:    {recall_score(y_test, xgb_preds, average='weighted', zero_division=0):.4f}")